# 05 — Classical Baseline (Binary MNIST 0 vs 1, 4 PCA Features)

**Purpose.** Provide a classical baseline for RQ5 (how does the hybrid
quantum-classical model compare to an equivalent classical model?). The
archived `02_classical_baseline.ipynb` trained on the *wrong* dataset
(full 10-class raw-pixel MNIST) and got 12% accuracy -- not usable for
this comparison. This notebook trains a lightweight feedforward network
on the *identical* binary 0-vs-1 / 4-PCA-feature data the VQC uses, with
the same train/val/test split methodology, so the comparison in RQ5 is
fair (same task, same features, same evaluation protocol).

Trained across the same 8 seeds planned for the VQC sweep (`04E`), since
classical training here is fast (seconds, not hours) -- this produces the
full statistical comparison set in one pass.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.seed import set_seed
from src.classical.baseline import ClassicalBaseline
from src.training.trainer import Trainer

print("Project root:", PROJECT_ROOT)

Project root: C:\Work\Quantum-Adversarial-Robustness


## Data

Same binary (0 vs 1) / 4-PCA-feature dataset used by the VQC pipeline.

In [2]:
X_train_full = np.load(PROJECT_ROOT / "data" / "binary" / "X_train.npy")
y_train_full = np.load(PROJECT_ROOT / "data" / "binary" / "y_train.npy")
X_test = np.load(PROJECT_ROOT / "data" / "binary" / "X_test.npy")
y_test = np.load(PROJECT_ROOT / "data" / "binary" / "y_test.npy")

print("X_train_full:", X_train_full.shape, " X_test:", X_test.shape)

X_train_full: (11824, 4)  X_test: (2956, 4)


## Multi-seed training

Same train/val split methodology as the VQC pipeline (80/20 stratified),
`StandardScaler` fit on train only, `Trainer` (the same class used for
the VQC) with `BCEWithLogitsLoss` + Adam. 30 epochs is generous given how
quickly this converges -- `Trainer` keeps the best-val-loss checkpoint.

In [3]:
SEEDS = [42, 43, 44, 45, 46, 47, 48, 49]

MODELS_DIR = PROJECT_ROOT / "results" / "models"
CLASSICAL_DIR = PROJECT_ROOT / "results" / "classical"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CLASSICAL_DIR.mkdir(parents=True, exist_ok=True)

all_results = []

for seed in SEEDS:
    set_seed(seed)

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=0.2, stratify=y_train_full, random_state=seed,
    )

    scaler = StandardScaler().fit(X_train)
    X_train_s = scaler.transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    def to_tensors(X, y):
        return (
            torch.tensor(X, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32).reshape(-1, 1),
        )

    X_train_t, y_train_t = to_tensors(X_train_s, y_train)
    X_val_t, y_val_t = to_tensors(X_val_s, y_val)
    X_test_t, y_test_t = to_tensors(X_test_s, y_test)

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_train_t, y_train_t),
        batch_size=32, shuffle=True,
    )
    val_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_val_t, y_val_t),
        batch_size=32, shuffle=False,
    )
    test_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_test_t, y_test_t),
        batch_size=32, shuffle=False,
    )

    model = ClassicalBaseline(in_features=4, hidden_dim=8)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.BCEWithLogitsLoss()

    trainer = Trainer(model, optimizer, criterion)
    trainer.fit(train_loader, val_loader, epochs=30)

    test_loss, test_metrics = trainer.evaluate(test_loader)

    n_params = sum(p.numel() for p in model.parameters())

    checkpoint_path = MODELS_DIR / f"classical_baseline_seed{seed}.pt"
    torch.save(model.state_dict(), checkpoint_path)

    result = {
        "seed": seed,
        "n_params": n_params,
        "test_loss": test_loss,
        "accuracy": test_metrics["accuracy"],
        "precision": test_metrics["precision"],
        "recall": test_metrics["recall"],
        "f1": test_metrics["f1"],
    }
    all_results.append(result)
    print(result)

results_df = pd.DataFrame(all_results)
results_df.to_csv(CLASSICAL_DIR / "classical_baseline_all_seeds.csv", index=False)
print()
print(results_df)

Epoch 001/030 | Train Loss: 0.0802 | Val Loss: 0.0088 | Train Acc: 0.9969 | Val Acc: 0.9966


Epoch 002/030 | Train Loss: 0.0083 | Val Loss: 0.0075 | Train Acc: 0.9976 | Val Acc: 0.9975


Epoch 003/030 | Train Loss: 0.0073 | Val Loss: 0.0073 | Train Acc: 0.9980 | Val Acc: 0.9970


Epoch 004/030 | Train Loss: 0.0065 | Val Loss: 0.0068 | Train Acc: 0.9982 | Val Acc: 0.9975


Epoch 005/030 | Train Loss: 0.0061 | Val Loss: 0.0071 | Train Acc: 0.9981 | Val Acc: 0.9970


...

Epoch 026/030 | Train Loss: 0.0065 | Val Loss: 0.0041 | Train Acc: 0.9980 | Val Acc: 0.9983


Epoch 027/030 | Train Loss: 0.0067 | Val Loss: 0.0044 | Train Acc: 0.9984 | Val Acc: 0.9979


Epoch 028/030 | Train Loss: 0.0065 | Val Loss: 0.0052 | Train Acc: 0.9981 | Val Acc: 0.9983


Epoch 029/030 | Train Loss: 0.0063 | Val Loss: 0.0046 | Train Acc: 0.9982 | Val Acc: 0.9979


Epoch 030/030 | Train Loss: 0.0065 | Val Loss: 0.0045 | Train Acc: 0.9983 | Val Acc: 0.9983
{'seed': 49, 'n_params': 49, 'test_loss': 0.004542062712463245, 'accuracy': 0.9979702300405954, 'precision': 0.9987285441830897, 'recall': 0.9974603174603175, 'f1': 0.9980940279542567}

   seed  n_params  test_loss  accuracy  precision    recall        f1
0    42        49   0.003503  0.998647   0.998730  0.998730  0.998730
1    43        49   0.003255  0.998309   0.997465  0.999365  0.998414
2    44        49   0.004048  0.998985   0.998731  0.999365  0.999048
3    45        49   0.002873  0.998647   0.998098  0.999365  0.998731
4    46        49   0.004021  0.998647   0.998730  0.998730  0.998730
5    47        49   0.003857  0.998647   0.998730  0.998730  0.998730
6    48        49   0.007805  0.996279   0.998724  0.994286  0.996500
7    49        49   0.004542  0.997970   0.998729  0.997460  0.998094


## Summary statistics across seeds

In [4]:
summary = results_df[["accuracy", "precision", "recall", "f1"]].agg(["mean", "std"])
print(summary)

summary.to_json(CLASSICAL_DIR / "classical_baseline_summary.json")

      accuracy  precision    recall        f1
mean  0.998266   0.998492  0.998254  0.998372
std   0.000857   0.000470  0.001722  0.000807


**Interpretation.** With `StandardScaler` correctly fit per-split and
`Trainer`'s best-val-loss checkpointing, this lightweight classical
network should land close to the ~99.66% ceiling already established by
plain `LogisticRegression`/`SVC(rbf)` sanity checks in the Phase 0
diagnostics (`04D`) -- this is the number RQ5 will compare the VQC's
~63-64% (Phase 0-accepted `single_z`/`ansatz_reps=4` configuration)
against.